In [2]:
import numpy as np
import tensorflow as tf

In [26]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / e_x.sum(axis=0, keepdims=True)


def lstm_cell_forward(xt, a_prev, c_prev, parameters):

    Wf = parameters["Wf"] 
    bf = parameters["bf"]
    Wi = parameters["Wi"] 
    bi = parameters["bi"] 
    Wc = parameters["Wc"] 
    bc = parameters["bc"]
    Wo = parameters["Wo"] 
    bo = parameters["bo"]
    Wy = parameters["Wy"] 
    by = parameters["by"]
    
    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    concat = np.concatenate((a_prev, xt), axis=0)

    ft = sigmoid(np.dot(Wf, concat) + bf)
    it = sigmoid(np.dot(Wi, concat) + bi)
    cct = np.tanh(np.dot(Wc, concat) + bc)
    c_next = ft * c_prev + it * cct
    ot = sigmoid(np.dot(Wo, concat) + bo)
    a_next = ot * np.tanh(c_next)
    
    #yt_pred = softmax(np.dot(Wy, a_next) + by)

    yt_pred = tf.keras.activations.softmax(np.dot(Wy, a_next) + by).numpy()
   
    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return a_next, c_next, yt_pred, cache

In [41]:
def cell_forward(x, c_prev, a_prev, params):

    forget_gate = tf.keras.activations.sigmoid(tf.linalg.matmul(params["Wf"], tf.concat(values=[a_prev, x], axis=0)) + params["bf"])
    update_gate = tf.keras.activations.sigmoid(tf.linalg.matmul(params["Wu"], tf.concat(values=[a_prev, x], axis=0)) + params["bu"])
    cell_state = tf.math.tanh(tf.linalg.matmul(params["Wc"], tf.concat(values=[a_prev, x], axis=0)) + params["bc"])
    c_next = forget_gate * c_prev + update_gate * cell_state
    output_gate = tf.keras.activations.sigmoid(tf.linalg.matmul(params["Wo"], tf.concat(values=[a_prev, x], axis=0)) + params["bo"])
    a_next = output_gate * tf.math.tanh(c_next)
    y_pred = tf.keras.activations.softmax(tf.linalg.matmul(params["Wy"], a_next) + params["by"])

    cache = (a_next, c_next, a_prev, c_prev, forget_gate, update_gate, cell_state, output_gate, x, params)

    return a_next, c_next, y_pred, cache

In [42]:
np.random.seed(1)
xt = np.random.randn(3, 10)
a_prev = np.random.randn(5, 10)
c_prev = np.random.randn(5, 10)


params = {
    "Wf": np.random.randn(5, 8),
    "bf": np.random.randn(5, 1),
    "Wi": np.random.randn(5, 8),
    "bi": np.random.randn(5, 1),
    "Wc": np.random.randn(5, 8),
    "bc": np.random.randn(5, 1),
    "Wo": np.random.randn(5, 8),
    "bo": np.random.randn(5, 1),
    "Wy": np.random.randn(2, 5),
    "by": np.random.randn(2, 1)
}


tf_params = {
    "Wf": tf.convert_to_tensor(params["Wf"], dtype=tf.float32),
    "bf": tf.convert_to_tensor(params["bf"], dtype=tf.float32),
    "Wu": tf.convert_to_tensor(params["Wi"], dtype=tf.float32),  
    "bu": tf.convert_to_tensor(params["bi"], dtype=tf.float32), 
    "Wc": tf.convert_to_tensor(params["Wc"], dtype=tf.float32),
    "bc": tf.convert_to_tensor(params["bc"], dtype=tf.float32),
    "Wo": tf.convert_to_tensor(params["Wo"], dtype=tf.float32),
    "bo": tf.convert_to_tensor(params["bo"], dtype=tf.float32),
    "Wy": tf.convert_to_tensor(params["Wy"], dtype=tf.float32),
    "by": tf.convert_to_tensor(params["by"], dtype=tf.float32)
}

x_tf = tf.convert_to_tensor(xt, dtype=tf.float32)
a_prev_tf = tf.convert_to_tensor(a_prev, dtype=tf.float32)
c_prev_tf = tf.convert_to_tensor(c_prev, dtype=tf.float32)

In [43]:
a_next_np, c_next_np, yt_pred_np, cache_np = lstm_cell_forward(xt, a_prev, c_prev, params)

a_next_tf, c_next_tf, y_pred_tf, cache_tf = cell_forward(x_tf, c_prev_tf, a_prev_tf, tf_params)


a_next_tf_np = a_next_tf.numpy()
c_next_tf_np = c_next_tf.numpy()
y_pred_tf_np = y_pred_tf.numpy()

print("Difference in a_next:", np.max(np.abs(a_next_np - a_next_tf_np)))
print("Difference in c_next:", np.max(np.abs(c_next_np - c_next_tf_np)))
print("Difference in yt_pred:", np.max(np.abs(yt_pred_np - y_pred_tf_np)))

Difference in a_next: 1.7448035749545454e-07
Difference in c_next: 1.8834472204076746e-07
Difference in yt_pred: 2.269626873108166e-07


In [45]:
def lstm_forward(x, a0, parameters):
  
    caches = []
    Wy = parameters['Wy'] 
    
    n_x, m, T_x = x.shape
    n_y, n_a = Wy.shape
    
    a = np.zeros((n_a, m, T_x), dtype=float)
    c = np.zeros((n_a, m, T_x), dtype=float)
    y = np.zeros((n_y, m, T_x), dtype=float)
    
    a_next = a0
    c_next = np.zeros((n_a, m), dtype=float)
    
    for t in range(T_x):
        
        xt = x[:,:,t]

        a_next, c_next, yt, cache = lstm_cell_forward(xt=xt, a_prev=a_next, c_prev=c_next, parameters=parameters)
        
        a[:,:,t] = a_next
        c[:,:,t]  = c_next
        y[:,:,t] = yt
        
        caches.append(cache)
        
    caches = (caches, x)

    return a, y, c, caches

In [48]:
def cells_forward(x, a0, params):
    
    caches = []
    n_x, m, t_x = x.shape
    n_y, n_a = params["Wy"].shape

    a = tf.TensorArray(dtype=tf.float32, size=t_x, clear_after_read=False)
    c = tf.TensorArray(dtype=tf.float32, size=t_x, clear_after_read=False)
    y = tf.TensorArray(dtype=tf.float32, size=t_x, clear_after_read=False)

   
    a_next = a0
    c_next = tf.zeros(shape=(n_a, m), dtype=tf.float32)

    for t in range(t_x):
        
        xt = x[:, :, t]
        a_next, c_next, yt, cache = cell_forward(x=xt, c_prev=c_next, a_prev=a_next, params=params)

        a = a.write(t, a_next)
        c = c.write(t, c_next)
        y = y.write(t, yt)

        caches.append(cache)
        
    a = tf.transpose(a.stack(), perm=[1, 2, 0])
    c = tf.transpose(c.stack(), perm=[1, 2, 0])
    y = tf.transpose(y.stack(), perm=[1, 2, 0])
    caches = (caches, x)

    return a, c, y, caches

In [49]:
np.random.seed(1)
xt = np.random.randn(3, 10, 5)
a0 = np.random.randn(5, 10)

params = {
    "Wf": np.random.randn(5, 8),
    "bf": np.random.randn(5, 1),
    "Wi": np.random.randn(5, 8),
    "bi": np.random.randn(5, 1),
    "Wc": np.random.randn(5, 8),
    "bc": np.random.randn(5, 1),
    "Wo": np.random.randn(5, 8),
    "bo": np.random.randn(5, 1),
    "Wy": np.random.randn(2, 5),
    "by": np.random.randn(2, 1)
}

tf_params = {
    "Wf": tf.convert_to_tensor(params["Wf"], dtype=tf.float32),
    "bf": tf.convert_to_tensor(params["bf"], dtype=tf.float32),
    "Wu": tf.convert_to_tensor(params["Wi"], dtype=tf.float32),
    "bu": tf.convert_to_tensor(params["bi"], dtype=tf.float32),
    "Wc": tf.convert_to_tensor(params["Wc"], dtype=tf.float32),
    "bc": tf.convert_to_tensor(params["bc"], dtype=tf.float32),
    "Wo": tf.convert_to_tensor(params["Wo"], dtype=tf.float32),
    "bo": tf.convert_to_tensor(params["bo"], dtype=tf.float32),
    "Wy": tf.convert_to_tensor(params["Wy"], dtype=tf.float32),
    "by": tf.convert_to_tensor(params["by"], dtype=tf.float32)
}

x_tf = tf.convert_to_tensor(xt, dtype=tf.float32)
a0_tf = tf.convert_to_tensor(a0, dtype=tf.float32)

a_np, y_np, c_np, caches_np = lstm_forward(xt, a0, params)
a_tf, c_tf, y_tf, caches_tf = cells_forward(x_tf, a0_tf, tf_params)

a_tf_np = a_tf.numpy()
c_tf_np = c_tf.numpy()
y_tf_np = y_tf.numpy()

print("Difference in a:", np.max(np.abs(a_np - a_tf_np)))
print("Difference in c:", np.max(np.abs(c_np - c_tf_np)))
print("Difference in y:", np.max(np.abs(y_np - y_tf_np)))

Difference in a: 2.5251595520137116e-07
Difference in c: 4.0442799997819634e-07
Difference in y: 2.0421451793484202e-08
